In [1]:
import os
import pickle
import gc
from tqdm import tqdm
import time
import polars as pl
import numpy as np
import shutil
import glob
from scipy import sparse
from tienlen_libs import Card, Utils, ActionGenerator, NaturalWin
from tienlen_libs.constant import ComboType
import math
from collections import defaultdict

In [2]:
def check_data(list_file_src_dir):
    list_who_play_first = []
    set_player1_card = set()
    for file_dir in tqdm(list_file_src_dir):
        try:
            with open(file_dir, 'rb') as f:
                data = pickle.load(f)
            list_who_play_first.append(data["who_play_first"])
            player1_card = tuple(data["player_card"][1])
            set_player1_card.add(player1_card)
            for state in data["Q_table"]:
                player_action = state[1]
                assert len(player_action) == 2
                player2_action = player_action[0]
                player3_action = player_action[1]
                turn = state[2]
                if turn == 1:
                    for player_id_action in player_action:
                        assert player_id_action == tuple() or player_id_action == -1
                elif turn == 2:
                    assert isinstance(player2_action, tuple) and len(player2_action) > 0
                    assert player3_action == tuple() or player3_action == -1
                elif turn == 3:
                    assert isinstance(player3_action, tuple) and len(player3_action) > 0

                player_card = state[0]
                for i in range(1, 4):
                    assert set(player_card[i-1]).issubset(set(data["player_card"][i]))
        except BaseException as e:
            print(f"ERROR with file {file_dir}")
            print(e)
    return list_who_play_first, set_player1_card

In [ ]:
def divide_all_files_data_into_3_part(all_files_data):
    print("Before shuffling the all_files_data")
    for i in range(5): print(all_files_data[i])
    np.random.shuffle(all_files_data)
    print("After shuffling the all_files_data")
    for i in range(5): print(all_files_data[i])
    #Divide the all_files_data into 3 parts: Part 1: 70%, Part 2: 20%, Part 3: 10%
    pos_part_1 = int(len(all_files_data)*0.7)
    pos_part_2 = int(len(all_files_data)*0.9)
    assert pos_part_1 < pos_part_2 < len(all_files_data)
    part_train = all_files_data[:pos_part_1]
    part_validation = all_files_data[pos_part_1:pos_part_2]
    part_test = all_files_data[pos_part_2:]
    print("The length of all_files_data: {:,}".format(len(all_files_data)))
    print("The length of part_train: {:,}".format(len(part_train)))
    print("The length of part_validation: {:,}".format(len(part_validation)))
    print("The length of part_test: {:,}".format(len(part_test)))
    assert len(part_train) + len(part_validation) + len(part_test) == len(all_files_data)
    return part_train, part_validation, part_test

def main_gen_train_validation_test_folder(data_all_dir, 
                                          data_train_dir, 
                                          data_validation_dir, 
                                          data_test_dir):
    for directory in [data_train_dir, data_validation_dir, data_test_dir]:
        os.makedirs(directory)

    all_files_data = glob.glob(os.path.join(data_all_dir, '*'))
    assert len(all_files_data) == len(os.listdir(data_all_dir))
    part_train, part_validation, part_test = divide_all_files_data_into_3_part(all_files_data)
    
    for part, part_dir in zip([part_train, part_validation, part_test], 
                            [data_train_dir, data_validation_dir, data_test_dir]):
        print(f"\nCopy the files to the {part_dir} folder...........")
        for src_dir in tqdm(part):
            file_name = os.path.basename(src_dir)
            des_dir = os.path.join(part_dir, file_name)
            assert not os.path.exists(des_dir)
            shutil.copy2(src_dir, des_dir)
            # shutil.copytree(src_dir, des_dir)

def divide_src_folder_into_multiple_small_folder(src_folder, files_per_small_folder=2600):
    print(f"\nSplitting the {src_folder} into multiple small folder")
    all_files_data = list(glob.glob(os.path.join(src_folder, '*')))
    np.random.shuffle(all_files_data)
    print(all_files_data[:5])
    total_files = len(all_files_data)
    n_small_folder = math.ceil(total_files / files_per_small_folder)
    
    for i in range(n_small_folder):
        small_folder_name = src_folder + f'_{i+1}'
        os.makedirs(small_folder_name)
        
        start_idx = i * files_per_small_folder
        end_idx = min(start_idx + files_per_small_folder, total_files)
        print(f"Moving files from index {start_idx} : {end_idx-1} to {small_folder_name}")
        for file_dir in all_files_data[start_idx:end_idx]:
            shutil.move(file_dir, small_folder_name)

data_all_dir = r"D:\Project\TLMN\data_see_opponent_card_3_players_preprocess_2"
data_train_dir = r"F:\dataset\data_see_opponent_card_3_players_train"
data_validation_dir = r"F:\dataset\data_see_opponent_card_3_players_validation"
data_test_dir = r"F:\dataset\data_see_opponent_card_3_players_test"

main_gen_train_validation_test_folder(data_all_dir, data_train_dir, data_validation_dir, data_test_dir)
divide_src_folder_into_multiple_small_folder(data_train_dir)
divide_src_folder_into_multiple_small_folder(data_validation_dir)
divide_src_folder_into_multiple_small_folder(data_test_dir)          

In [ ]:
import polars as pl
import os

folder_data = r"F:\dataset\data_see_opponent_card_3_players_train_1"
output_file_dir = folder_data + ".parquet"
print(f"Merging the folder data {folder_data}")
df = pl.scan_parquet(os.path.join(folder_data, "*.parquet"), low_memory=True)
print(df)
print(f"Shuffling the merge data ....................")
df = df.collect().sample(fraction=1.0, shuffle=True)
print(f"Shuffling DONE => Writing to {output_file_dir}")
df.write_parquet(output_file_dir)
print(f"Writing DONE")

In [ ]:
def check_range_value_of_a_series(column_series, min_value=0, max_value=1):
    assert (min_value <= column_series).all() and (column_series <= max_value).all(), \
        f"The range value of the column {column_series.name} is not correct"

def check_range_value_after_min_max_scale(dataset):
    for col_name in dataset.columns:
        check_range_value_of_a_series(dataset[col_name])

def normalize_dataset(dataset):
    bool_cols = [col for col in dataset.columns if dataset[col].dtype == pl.Boolean]
    dataset = dataset.with_columns(
        [
            pl.col(bool_cols).cast(pl.Int8),  # Convert boolean columns to Int8
            (pl.col("player2_action_max_card") + 1) / 52,  # Normalize player2_action_max_card
            (pl.col("player3_action_max_card") + 1) / 52,  # Normalize player3_action_max_card
            (pl.col("action_max_card") + 1) / 52,  # Normalize action_max_card
        ]
    )
    return dataset

src_data = r"F:\dataset"
batch_size = 4096 * 16
print(f"batch_size = {batch_size:,}")
folder_parent_dir = r"D:\Project\TLMN\big_data"

if not os.path.exists(folder_parent_dir):
    os.makedirs(folder_parent_dir)

for dataset_type in ["validation"]:
    count = 0
    folder_dataset_type = os.path.join(folder_parent_dir, f"data_{dataset_type}")
    os.makedirs(folder_dataset_type)
    print(f"\nLoading the {dataset_type} data")
    dataset_dir = os.path.join(src_data, f"data_{dataset_type}")
    for file_dataset_dir in glob.glob(os.path.join(dataset_dir, "*.parquet")):
        dataset = pl.read_parquet(file_dataset_dir)
        # len_dataset = dataset.select(pl.len()).collect().item()
        len_dataset = len(dataset)
        print(f"The length of dataset {dataset_type} from {file_dataset_dir} is {len_dataset:,}")
        for i in tqdm(range(0, len_dataset, batch_size)):
            # batch_data = dataset.slice(i, batch_size).collect()
            batch_data = dataset.slice(i, batch_size)
            batch_data.write_parquet(os.path.join(folder_dataset_type, f"data_{dataset_type}_batch_{count}.parquet"))
            count += 1
        del dataset, batch_data; gc.collect()